# Prompt Piper — Real Experiment: GPT-4o × GSM8K

A complete end-to-end experiment using GPT-4o via the OpenAI API:

* **Task**: `gsm8k` — grade-school math, generative scoring (`generate_until`).
* **Model**: `gpt-4o` via OpenAI Chat Completions API.
* **Fitting**: `DatasetVocabularyFitter` mines the training split; LLM-backed
  operators (O4, O10, O15, O28) call GPT-4o for pattern discovery.
* **Search**: NSGA-II, pop=50, 100 generations, n_subset=100.
* **Comparison**: fitted factory vs default factory on the same config.
* **Held-out eval**: Pareto solutions re-scored on a test slice.

> **API cost notice**: each optimisation run makes ≈ 5 050 GPT-4o API calls
> (one per solution × n_subset prompts each).  The fitting phase adds
> O(dozens) of extra calls for pattern discovery when `use_llm_for_semantic=True`.
> Set `pop_size=10, n_gen=20, n_subset=20` for a cheap smoke test.
> Requires `OPENAI_API_KEY` in the environment.

**Differences from the HF notebook**
- No GPU, no quantisation config.
- `model_type='openai-chat-completions'`; the OpenAI API handles chat
  template formatting internally.
- Only `generate_until` tasks work — loglikelihood (ARC, MMLU, GPQA) is not
  supported by the chat completions API.
- `batch_size=1` (chat completions API processes one request at a time).

**Sections**
1. Load model and datasets
2. Inspect a GSM8K sample
3. Fit vocabulary and patterns with GPT-4o
4. Build combined factory
5. Baseline accuracy
6. Run optimisation — fitted factory
7. Run optimisation — default factory (comparison)
8. Pareto front: fitted vs default
9. Apply best pipeline to a sample
10. Held-out test evaluation
11. Operator frequency analysis

In [1]:
import sys
sys.path.insert(0, '../src')
from dotenv import load_dotenv
load_dotenv()

import warnings
warnings.filterwarnings('ignore')

import os
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

pd.set_option('display.max_colwidth', 120)

if not os.environ.get('OPENAI_API_KEY'):
    raise EnvironmentError(
        'OPENAI_API_KEY is not set. '
        'Export it before starting the notebook or add it to your .env file.'
    )

In [2]:
MODEL_NAME = 'gpt-4o'       # change to 'gpt-4o-mini' for a cheaper run
NUM_FS     = 3              # few-shot examples
BS         = 1              # batch_size — chat API is always 1

## 1. Load model and datasets

GSM8K has two splits:
- `train` (~7 500 samples) — used exclusively for fitting.
- `test` (~1 319 samples) — first 1 000 for search, last ~319 held out.

`LMEvalBackend.build` with `model_type='openai-chat-completions'` loads
lm-eval's `OpenAIChatCompletion` wrapper around the OpenAI chat API.
There is no GPU allocation and no tokeniser download — the backend
counts tokens via `tiktoken` (the same library OpenAI uses internally).

**Chat template**: the OpenAI API handles message formatting internally;
do **not** pass `apply_chat_template=True` here.

**Stop sequences**: GSM8K's lm-eval config ships with
`until=["Question:", "</s>"]`.  For GPT-4o the `</s>` token is harmless
(it will never appear in the generation), and `"Question:"` cleanly
terminates chain-of-thought responses.  No `extra_stop_sequences` needed.

In [3]:
from prompt_piper.backends.lm_eval import LMEvalBackend

backend = LMEvalBackend.build(
    model_type='openai-chat-completions',
    model_name=MODEL_NAME,
    task_name='gsm8k',
    num_fewshot=NUM_FS,
    metric_key='exact_match',
    batch_size=BS,
    max_gen_toks=1024,   # GSM8K needs room for chain-of-thought
)

print(f'Task output type : {getattr(backend.task, "OUTPUT_TYPE", "unknown")}')
print(f'Tokenizer type   : {type(backend.tokenizer._counter).__name__}')
print()
print('Note: batch_size=1 is enforced by the chat completions API.')
print('Each model call issues one HTTP request to api.openai.com.')

Task output type : generate_until
Tokenizer type   : function

Note: batch_size=1 is enforced by the chat completions API.
Each model call issues one HTTP request to api.openai.com.


In [4]:
# Training split — for vocabulary fitting only.
# template_field='prompt_0shot' stores the 0-shot question alongside the
# full prompt so the fitter can detect the few-shot boundary.
train_data = backend.get_dataset(
    split='train',
    num_fewshot=NUM_FS,
    prompt_field='prompt',
    template_field='prompt_0shot',
).as_list()

# Test split — first 1 000 for search, last ~319 held out.
test_data = backend.get_dataset(
    split='test',
    num_fewshot=NUM_FS,
    prompt_field='prompt',
).as_list()

search_data  = test_data[:1000]
heldout_data = test_data[1000:]

print(f'Train samples   : {len(train_data)}')
print(f'Search samples  : {len(search_data)}')
print(f'Held-out samples: {len(heldout_data)}')

Train samples   : 7473
Search samples  : 1000
Held-out samples: 319


## 2. Inspect a GSM8K sample

In [5]:
sample = train_data[0]
print('Keys:', list(sample.keys()))
print()
print('=== Prompt ===')
print(sample['prompt'])
print()
print(f'=== Target: {sample["target"]!r} ===')
print()
tok_count = backend.tokenizer.count(sample['prompt'])
print(f'Token count (tiktoken / {MODEL_NAME}): {tok_count}')

counts = [backend.tokenizer.count(s['prompt']) for s in train_data[:200]]
print(f'Token counts over first 200 train samples — '
      f'min={min(counts)}, median={int(np.median(counts))}, max={max(counts)}')

Keys: ['prompt', 'target', '_doc', 'prompt_0shot']

=== Prompt ===
Question: Jen and Tyler are gymnasts practicing flips. Jen is practicing the triple-flip while Tyler is practicing the double-flip. Jen did sixteen triple-flips during practice. Tyler flipped in the air half the number of times Jen did. How many double-flips did Tyler do?
Answer: Jen did 16 triple-flips, so she did 16 * 3 = <<16*3=48>>48 flips.
Tyler did half the number of flips, so he did 48 / 2 = <<48/2=24>>24 flips.
A double flip has two flips, so Tyler did 24 / 2 = <<24/2=12>>12 double-flips.
#### 12

Question: Four people in a law firm are planning a party. Mary will buy a platter of pasta for $20 and a loaf of bread for $2. Elle and Andrea will split the cost for buying 4 cans of soda which cost $1.50 each, and chicken wings for $10. Joe will buy a cake that costs $5. How much more will Mary spend than the rest of the firm put together?
Answer: Mary will spend $20 + $2 = $<<20+2=22>>22.
Elle and Andrea will spend 

## 3. Fit vocabulary and patterns with GPT-4o

`DatasetVocabularyFitter` analyses the training corpus.  LLM-backed operators
(O4, O10, O15, O28) call GPT-4o for pattern discovery.

> **Cost**: fitting on 500 samples triggers O(dozens) of GPT-4o API calls.
> Reduce `sample_size` or set `backend=None` to run corpus-only fitting at
> no API cost.

In [6]:
from prompt_piper.transformations.fitting import DatasetVocabularyFitter

fitter = DatasetVocabularyFitter(
    train_data,
    prompt_field='prompt',
    target_field='target',
    template_field='prompt_0shot',
    compress_field='prompt',
    sample_size=500,
    seed=42,
    backend=backend.semantic_backend,  # GPT-4o for O4, O10, O15, O28
)

print('Fitting vocabulary and patterns on 500 training samples...')
report = fitter.report()
print('Done.')

Fitting vocabulary and patterns on 500 training samples...


Requesting API: 100%|██████████| 1/1 [00:00<00:00,  1.96it/s]

Done.


In [7]:
print('=== New stopwords (beyond defaults) ===')
print(', '.join(sorted(report['stopwords'])) or '  (none)')

print()
print('=== New abbreviations ===')
for pat, repl in report['abbreviations']:
    print(f'  {pat!r:55s}  ->  {repl!r}')
if not report['abbreviations']:
    print('  (none beyond defaults)')

print()
print('=== New section headings ===')
print(', '.join(repr(s) for s in report['sections']) or '  (none)')

if 'short_synonyms' in report:
    print()
    print('=== Short synonyms (GPT-4o-discovered) ===')
    for pat, repl in report['short_synonyms']:
        print(f'  {pat!r:55s}  ->  {repl!r}')

if 'imperative_patterns' in report:
    print()
    print('=== Imperative-rewrite patterns (GPT-4o-discovered) ===')
    for p in report['imperative_patterns']:
        print(f'  {p!r}')

if 'chunk_separator' in report:
    print()
    print(f'=== Chunk separator (GPT-4o-discovered) ===')
    print(f'  {report["chunk_separator"]!r}')

=== New stopwords (beyond defaults) ===
  (none)

=== New abbreviations ===
  (none beyond defaults)

=== New section headings ===
  (none)

=== Short synonyms (GPT-4o-discovered) ===
  '\\bpounds\\ of\\ ground\\ mince\\b'                     ->  'pounds'
  '\\bin\\ total\\b'                                       ->  'total'
  '\\btherefore\\b'                                        ->  'so'
  '\\ba\\ total\\ of\\b'                                   ->  'total'
  '\\bthis\\ means\\b'                                     ->  'so'
  '\\ba\\ total\\ of\\b'                                   ->  'total'
  '\\ba\\ total\\ of\\b'                                   ->  'total'
  '\\ba\\ total\\ of\\b'                                   ->  'total'
  '\\ba\\ total\\ of\\b'                                   ->  'total'
  '\\ba\\ total\\ of\\b'                                   ->  'total'
  '\\ba\\ total\\ of\\b'                                   ->  'total'
  '\\ba\\ total\\ of\\b'                

## 4. Build combined factory

Same pattern as the HF notebook: corpus/LLM-fitted instances take priority;
remaining semantic operators receive the GPT-4o backend.

In [8]:
fitted_instances    = fitter.fit_all()
llm_semantic_backend = backend.semantic_backend

def combined_factory(cls):
    if cls.id in fitted_instances:
        return fitted_instances[cls.id]
    if (cls.category == 'semantic'
            and 'backend' in inspect.signature(cls.__init__).parameters):
        return cls(backend=llm_semantic_backend)
    return cls()

print('Fitted operator IDs:', sorted(fitted_instances.keys()))

Requesting API: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]

Fitted operator IDs: ['O10', 'O15', 'O17', 'O18', 'O21', 'O22', 'O28', 'O3', 'O4', 'O5']


## 5. Baseline accuracy

Measure exact-match accuracy on uncompressed prompts.  This is the upper
bound that compression should try to preserve.

> **Cost**: 50 GPT-4o API calls (one per sample).

In [9]:
baseline_samples = heldout_data[:50]
originals = [s['prompt'] for s in baseline_samples]

scores = backend.evaluator.evaluate_batch(originals, originals, baseline_samples)
baseline_acc = float(np.mean(scores))

print(f'Baseline exact-match accuracy (n=50, uncompressed): {baseline_acc:.3f}')
print(f'(Typical range for GPT-4o on {NUM_FS}-shot GSM8K: 0.85 - 0.95)')

Requesting API: 100%|██████████| 50/50 [02:13<00:00,  2.68s/it]

Baseline exact-match accuracy (n=50, uncompressed): 0.420
(Typical range for GPT-4o on 3-shot GSM8K: 0.85 - 0.95)


## 6. Run optimisation — fitted factory

Search budget:

| | Value | API calls |
|---|---|---|
| Initial population | pop_size = 50 | 50 × 100 = 5 000 |
| 100 generations | 50 offspring/gen | 5 000 × 100 = 500 000 |
| **Total sample evals** | | **~505 000** |
| **GPT-4o API calls** | | **5 050** |

> **Cost estimate**: at \$5/1M input tokens and ~200 tokens/prompt,
> 505 000 evals ≈ ~\$500 for the full run.  Use `pop_size=10, n_gen=20`
> for a ~25× cheaper preliminary run (~\$20).

In [10]:
from prompt_piper.optimization import ExperimentConfig

config = ExperimentConfig(
    algorithm='nsga2',
    pop_size=50,
    n_gen=20,
    n_subset=50,
    subset_seed=42,
    seed=42,
)

total_evals = config.pop_size * (1 + config.n_gen)
print(f'Individual evaluations : {total_evals:,}')
print(f'Sample evaluations     : {total_evals * config.n_subset:,}')
print(f'Estimated GPT-4o calls : {total_evals:,} (one per solution × n_subset prompts each)')

Individual evaluations : 1,050
Sample evaluations     : 52,500
Estimated GPT-4o calls : 1,050 (one per solution × n_subset prompts each)


In [ ]:
from prompt_piper.optimization.runner import Experiment

print('Running NSGA-II with fitted factory...')
experiment_fitted = Experiment(
    config=config,
    dataset=search_data,
    evaluator=backend.evaluator,
    tokenizer=backend.tokenizer,
    transformation_factory=combined_factory,
)
result_fitted = experiment_fitted.run(
    baselines=['no_compression', 'all_transformations']
)
print(result_fitted.summary())

Running NSGA-II with fitted factory...


Requesting API:   0%|          | 0/1 [00:00<?, ?it/s]

## 7. Run optimisation — default factory

In [ ]:
print('Running NSGA-II with default factory...')
experiment_default = Experiment(
    config=config,
    dataset=search_data,
    evaluator=backend.evaluator,
    tokenizer=backend.tokenizer,
)
result_default = experiment_default.run(
    baselines=['no_compression', 'all_transformations']
)
print(result_default.summary())

## 8. Pareto front: fitted vs default

In [ ]:
def pareto_df(result, label):
    rows = []
    for bits, names, tokens, perf in zip(
        result.pareto_solutions,
        result.pareto_transformation_names,
        result.avg_tokens,
        result.avg_performance,
    ):
        rows.append({
            'factory':    label,
            'avg_tokens': round(float(tokens), 1),
            'accuracy':   round(float(perf), 3),
            'n_ops':      int(bits.sum()),
            'operators':  ', '.join(names) if names else '(none)',
        })
    return pd.DataFrame(rows).sort_values('avg_tokens')

df_fitted  = pareto_df(result_fitted,  'fitted')
df_default = pareto_df(result_default, 'default')

print('=== Fitted factory Pareto front ===')
display(df_fitted[['avg_tokens', 'accuracy', 'n_ops', 'operators']])
print()
print('=== Default factory Pareto front ===')
display(df_default[['avg_tokens', 'accuracy', 'n_ops', 'operators']])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for res, label, color in [
    (result_fitted,  'Fitted factory (NSGA-II)',  'steelblue'),
    (result_default, 'Default factory (NSGA-II)', 'darkorange'),
]:
    idx = res.avg_tokens.argsort()
    ax.plot(res.avg_tokens[idx], res.avg_performance[idx],
            'o-', label=label, zorder=3)

nc = result_fitted.baselines['no_compression']
at = result_fitted.baselines['all_transformations']
ax.scatter(nc.avg_tokens.mean(), nc.avg_performance.mean(),
           marker='*', s=250, color='gray', zorder=5, label='no_compression')
ax.scatter(at.avg_tokens.mean(), at.avg_performance.mean(),
           marker='D', s=80, color='red',  zorder=5, label='all_transformations')
ax.axhline(baseline_acc, color='gray', linestyle='--',
           linewidth=0.8, label=f'baseline acc ({baseline_acc:.3f})')

ax.set_xlabel('Average tokens (compressed prompt)')
ax.set_ylabel('Exact-match accuracy (GSM8K)')
ax.set_title(f'{MODEL_NAME} on GSM8K — Fitted vs Default factory')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

nc_tok = float(nc.avg_tokens.mean())
_, tok_f, perf_f = result_fitted.best_compression()
_, tok_d, perf_d = result_default.best_compression()
print(f'No compression            : {nc_tok:.0f} tokens, acc={baseline_acc:.3f}')
print(f'Best compression (fitted) : {tok_f:.0f} tokens ({tok_f/nc_tok:.1%}), acc={perf_f:.3f}')
print(f'Best compression (default): {tok_d:.0f} tokens ({tok_d/nc_tok:.1%}), acc={perf_d:.3f}')

## 9. Apply best pipeline to a sample

In [ ]:
from prompt_piper.transformations import ALL_TRANSFORMATIONS

bits_comp, tok_comp, perf_comp = result_fitted.best_compression()
bits_perf, tok_perf, perf_perf = result_fitted.best_performance()

print(f'Best-compression solution : {tok_comp:.0f} avg tokens, acc={perf_comp:.3f}')
print(f'Best-performance solution : {tok_perf:.0f} avg tokens, acc={perf_perf:.3f}')

all_instances = [combined_factory(cls) for cls in ALL_TRANSFORMATIONS]
pipeline = [inst for inst, b in zip(all_instances, bits_comp) if b]

print(f'\nOperators in best-compression pipeline ({len(pipeline)}):')
for inst in pipeline:
    print(f'  {inst.name}')

In [ ]:
sample = heldout_data[5]
original = sample['prompt']

compressed = original
for op in pipeline:
    compressed = op(compressed)

n_orig = backend.tokenizer.count(original)
n_comp = backend.tokenizer.count(compressed)

print(f'=== Original ({n_orig} tokens) ===')
print(original)
print()
print(f'=== Compressed ({n_comp} tokens, {n_comp/n_orig:.1%} of original) ===')
print(compressed)

In [ ]:
eval_samples = heldout_data[:20]
orig_texts   = [s['prompt'] for s in eval_samples]
comp_texts   = []
for t in orig_texts:
    c = t
    for op in pipeline:
        c = op(c)
    comp_texts.append(c)

orig_scores = backend.evaluator.evaluate_batch(orig_texts, orig_texts, eval_samples)
comp_scores = backend.evaluator.evaluate_batch(orig_texts, comp_texts, eval_samples)

print(f'Accuracy over 20 held-out samples')
print(f'  Original  : {np.mean(orig_scores):.3f}')
print(f'  Compressed: {np.mean(comp_scores):.3f}')

rows = []
for ot, ct, os_, cs_ in zip(orig_texts, comp_texts, orig_scores, comp_scores):
    rows.append({
        'orig_tok': backend.tokenizer.count(ot),
        'comp_tok': backend.tokenizer.count(ct),
        'orig_acc': int(os_),
        'comp_acc': int(cs_),
    })
display(pd.DataFrame(rows))

## 10. Held-out test evaluation

Re-score every Pareto solution on `heldout_data` (the 319 samples not seen
during search).  A small positive gap (search > held-out) is normal; a large
gap suggests overfitting to the search subset.

In [ ]:
print(f'Re-scoring {len(result_fitted.pareto_solutions)} Pareto solutions on '
      f'{len(heldout_data)} held-out samples...')

heldout_objectives = experiment_fitted.evaluate_solutions(
    result_fitted.pareto_solutions,
    heldout_data,
)

tokens_heldout = heldout_objectives[:, 0]
perf_heldout   = heldout_objectives[:, 1]
print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

idx = result_fitted.avg_tokens.argsort()
ax.plot(result_fitted.avg_tokens[idx], result_fitted.avg_performance[idx],
        'o--', color='steelblue', alpha=0.5,
        label=f'Search phase (n_subset={config.n_subset})')

idx2 = tokens_heldout.argsort()
ax.plot(tokens_heldout[idx2], perf_heldout[idx2],
        'o-', color='steelblue',
        label=f'Held-out test (n={len(heldout_data)})')

ax.scatter(nc.avg_tokens.mean(), nc.avg_performance.mean(),
           marker='*', s=250, color='gray', zorder=5, label='no_compression')
ax.axhline(baseline_acc, color='gray', linestyle='--', linewidth=0.8,
           label=f'baseline acc ({baseline_acc:.3f})')

ax.set_xlabel('Average tokens')
ax.set_ylabel('Exact-match accuracy (GSM8K)')
ax.set_title(f'Search-phase vs held-out — {MODEL_NAME} / GSM8K')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

gap = float(np.mean(result_fitted.avg_performance) - np.mean(perf_heldout))
print(f'Mean accuracy — search: {np.mean(result_fitted.avg_performance):.3f}  '
      f'held-out: {np.mean(perf_heldout):.3f}  gap: {gap:+.3f}')

## 11. Operator frequency analysis

In [ ]:
counts_fitted  = Counter()
counts_default = Counter()

for names in result_fitted.pareto_transformation_names:
    counts_fitted.update(names)
for names in result_default.pareto_transformation_names:
    counts_default.update(names)

all_ops = sorted(set(counts_fitted) | set(counts_default))

if all_ops:
    fig, axes = plt.subplots(
        1, 2,
        figsize=(14, max(4, len(all_ops) * 0.4 + 1)),
        sharey=True,
    )
    for ax, counts, label, color in [
        (axes[0], counts_fitted,  'Fitted factory',  'steelblue'),
        (axes[1], counts_default, 'Default factory', 'darkorange'),
    ]:
        if counts:
            ops, freqs = zip(*counts.most_common())
            ax.barh(list(ops)[::-1], list(freqs)[::-1], color=color)
        ax.set_xlabel('Frequency on Pareto front')
        ax.set_title(label)
    plt.suptitle(f'Operator frequency — {MODEL_NAME} / GSM8K', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No operators selected on the Pareto front.')

only_fitted  = set(counts_fitted)  - set(counts_default)
only_default = set(counts_default) - set(counts_fitted)
if only_fitted:
    print('Operators unique to fitted front  :', ', '.join(sorted(only_fitted)))
if only_default:
    print('Operators unique to default front :', ', '.join(sorted(only_default)))